In [1]:
import csv
import re
from dataclasses import asdict, dataclass, fields
from enum import StrEnum, auto

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

In [2]:
class Stage(StrEnum):
    GROUP = auto()
    R32 = auto()
    R16 = auto()
    QUARTER = auto()
    SEMI = auto()
    THIRD = auto()
    FINAL = auto()

@dataclass
class Tip:
    name: str
    player_id: str
    stage: Stage
    home: str
    away: str
    h_score: str
    a_score: str
    
    @property
    def match_identifier(self):
        return f"{self.home.replace(' ', '_')}-{self.away.replace(' ', '_')}"

In [3]:
driver = webdriver.Chrome()
try:
    driver.get("https://wc2026-tipping-nu.vercel.app/leaderboard")
    WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CLASS_NAME, "lb-row"))
        )
    
    tips = []
    
    rows = driver.find_elements(by=By.CLASS_NAME, value="lb-row")
    
    for row in rows:    
        name = row.find_element(by=By.CLASS_NAME, value="lb-name").text
        row_button = row.find_element(by=By.TAG_NAME, value="button")
        on_click = row_button.get_attribute("onClick")
        player_id = re.match(R"viewPlayerTips\('(.+)'\)", on_click).group(1)
        
        row_button.click()
        
        view_modal = driver.find_element(by=By.CLASS_NAME, value="view-modal-box")
        
       
        group_content = view_modal.find_element(by=By.ID, value="vmcontent-group")
        
        tips.extend(
            Tip(stage=Stage.GROUP, name=name, player_id=player_id, home=home, away=away, h_score=h_score, a_score=a_score)
            for home, h_score, a_score, away in re.findall(R"(.+)\n(\d+) – (\d+)\n(.+)", group_content.text)
        )
        
        bracket_button = view_modal.find_element(by=By.ID, value="vmtab-bracket")
        bracket_button.click()
        
        bracket_content = view_modal.find_element(by=By.ID, value="vmcontent-bracket")
        
        stage_contents = bracket_content.find_elements(By.XPATH, "./*")
        for stage_content in stage_contents:
            stage_raw = stage_content.text.splitlines()[0].lower()
            match stage_raw:
                case "round of 32":
                    stage = Stage.R32
                case "round of 16":
                    stage = Stage.R16
                case "quarter finals":
                    stage = Stage.QUARTER
                case "semi finals":
                    stage = Stage.SEMI
                case "3rd place play-off":
                    stage = Stage.THIRD
                case "final":
                    stage = Stage.FINAL
            tips.extend(
                Tip(stage=stage, name=name, player_id=player_id, home=home, away=away, h_score=h_score, a_score=a_score)
                for home, h_score, a_score, away in re.findall(R"(.+)\n(\d+) – (\d+)\n(.+)", stage_content.text)
            )
        close_button = view_modal.find_element(by=By.CLASS_NAME, value="view-modal-close")
        close_button.click()
finally:
    driver.quit()

In [5]:
with open('01_scraped.csv', 'w') as f:
   w = csv.DictWriter(f, [fld.name for fld in fields(Tip)])
   w.writeheader()
   w.writerows(asdict(tip) for tip in tips)
